In [0]:
import pandas as pd
import datetime as dt
import pyspark.sql.functions as F

df = spark.read.csv("/Volumes/workspace/default/ev_data/ev_sessions_raw.csv", header=True, inferSchema=True)

def check_no_duplication_session(df):
    dupes = df.count() - df.select("session_id").distinct().count()
    return{"rule": "unique session_id", "failed_rows": dupes, "passed":dupes == 0, "label": "Critical"}

def check_no_future_date(df,as_of_date = dt.date.today()):
    future = df.filter(df.session_date > as_of_date).count()
    return{"rule": "no future date", "failed_rows": future, "passed":future == 0, "label": "Critical"}

def check_positive_revenue(df):
    neg_rev = df.filter(df.revenue_gbp < 0).count()
    return{"rule": "no negative revenue", "failed_rows": neg_rev, "passed":neg_rev == 0, "label": "Critical"}

def check_valid_status(df): 
    valid_status = df.filter(~df.charger_status.isin(['completed', 'failed', 'interrupted'])).count()
    return {"rule": "valid status", "failed_rows": valid_status, "passed": valid_status == 0, "label": "Warning"}

def check_no_null_critical(df):
    nulls = df.filter(df.session_date.isNull() | df.session_id.isNull() | df.kwh_delivered.isNull()).count()
    return {"rule": "no nulls", "failed_rows": nulls, "passed": nulls == 0, "label": "Critical"}

def check_postive_kwh(df):
    neg_kwh = df.filter(df.kwh_delivered <0).count()
    return {"rule": "no negative kwh", "failed_rows": neg_kwh, "passed": neg_kwh <= 0, "label": "Warning"}

#display (check_postive_kwh(df))

rules = [check_no_duplication_session,check_no_future_date,check_positive_revenue, check_valid_status, check_no_null_critical,check_postive_kwh]
results = []
for rule in rules:
    ruls_result = rule(df) #run rule to get result
    if ruls_result["passed"] == "false" and ruls_result["label"] == "Critical":
        raise Exception("Rule failed")
    results.append(rule(df))



final_result = spark.createDataFrame(results)
final_table = final_result.withColumn("date",F.current_timestamp())
display(final_table)

# final_table.write.format("delta").mode("append").saveAsTable("rule_check")

# spark.sql("select * from rule_check").display()

print(f"{DQ Summary}"

: 